# Topic Modeling (BERTopic)

Discovers latent topics in a text column using BERTopic with sentence-transformers embeddings. Adds `Topic#number` and `Topic_Label` columns. Shows a per-topic keyword table and representative documents.

BERTopic runs locally — no API key required. First run will download the embedding model (~90 MB).

A free HuggingFace token speeds up the model download. Get one at https://huggingface.co/settings/tokens

In [ ]:
# ── Load SuAVE parameters ──────────────────────────────────────────────────
import sys, os, requests as _req
from urllib.parse import urlparse as _urlparse
sys.path.insert(0, '../../helpers')
import suave_utils as su
from IPython.display import display, Markdown
import pandas as pd

def printmd(string):
    display(Markdown(string))

SUAVE_TOKEN = ''   # @param {type:"string"}
SUAVE_HOST  = ''   # @param {type:"string"}

_p = su.load_params(token=SUAVE_TOKEN, host=SUAVE_HOST)
if _p is None:
    raise RuntimeError('No SuAVE params. Open via SuAVEDispatch.ipynb, or enter token above.')

user          = _p.get('user', '')
survey_url    = _p.get('surveyurl', '')
csv_file      = _p.get('csv', '')
dzc_file      = _p.get('dzc', '')
active_object = _p.get('activeobject', 'none')
params        = ''
_caps         = su.detect_capabilities(_p)
views         = ','.join(_caps.get('views', []))
view          = 'grid'
_prefix       = os.environ.get('JUPYTERHUB_SERVICE_PREFIX', '/')
full_notebook_url = _prefix.rstrip('/') + '/lab/tree/SuAVEDispatch.ipynb'

localdzc             = _caps.get('localdzc', '')
full_images          = _caps.get('full_images', '')
full_images_location = full_images

absolutePath = os.path.expanduser('~/temp_csvs/')
os.makedirs(absolutePath, exist_ok=True)
_origin = _urlparse(survey_url)
_csv_url = f"{_origin.scheme}://{_origin.netloc}/surveys/{csv_file}"
_r = _req.get(_csv_url, timeout=30, verify=False)
_r.raise_for_status()
with open(absolutePath + csv_file, 'wb') as _f:
    _f.write(_r.content)

su.show_params(_p)

In [ ]:
# Install BERTopic if not present (takes ~2 minutes on first run)
try:
    import bertopic
except ImportError:
    import subprocess, sys as _sys
    print('Installing BERTopic...')
    subprocess.check_call([_sys.executable, '-m', 'pip', 'install', '-q',
                           'bertopic', 'sentence-transformers'])
    print('Done.')

In [ ]:
import sys
sys.path.insert(1, '../../helpers')
import panel_libs as panellibs
import suave_integration as suaveint

import ipywidgets as widgets
import pandas as pd
import numpy as np

## 1. Load data and configure topic model

In [ ]:
df = panellibs.extract_data(absolutePath + csv_file)
print(f"Loaded {len(df)} rows")

text_cols = [c for c in df.columns if '#number' not in c and '#img' not in c
             and '#hidden' not in c and '#date' not in c]

col_selector = widgets.Dropdown(
    options=text_cols, description='Text column:',
    layout=widgets.Layout(width='60%')
)
min_topic = widgets.IntSlider(
    value=10, min=3, max=100, description='Min topic size:',
    continuous_update=False
)
n_topics = widgets.IntText(
    value=0, description='# topics (0=auto):',
    layout=widgets.Layout(width='40%')
)
display(col_selector, min_topic, n_topics)

## 2. Fit BERTopic

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

col     = col_selector.value
nr_top  = n_topics.value if n_topics.value > 0 else None

docs = df[col].fillna('').astype(str).tolist()

printmd('**Loading embedding model (all-MiniLM-L6-v2)...**')
embedder = SentenceTransformer('all-MiniLM-L6-v2')

topic_model = BERTopic(
    embedding_model=embedder,
    min_topic_size=min_topic.value,
    nr_topics=nr_top,
    verbose=True
)

printmd('**Fitting topic model...**')
topics, probs = topic_model.fit_transform(docs)

df['Topic#number'] = topics

# Build human-readable label from top 3 keywords
topic_info = topic_model.get_topic_info()
label_map = {}
for _, row in topic_info.iterrows():
    tid = row['Topic']
    if tid == -1:
        label_map[tid] = 'Outlier'
    else:
        kws = topic_model.get_topic(tid)
        label_map[tid] = ', '.join([w for w, _ in kws[:3]])

df['Topic_Label'] = df['Topic#number'].map(label_map)

n_topics_found = len(set(topics)) - (1 if -1 in topics else 0)
n_outliers     = (pd.Series(topics) == -1).sum()
printmd(f"**Found {n_topics_found} topics; {n_outliers} outlier rows (Topic -1).**")

## 3. Topic keyword table and representative documents

In [ ]:
printmd('**Topic overview (top 10):**')
display(topic_info[topic_info['Topic'] != -1].head(10)[['Topic', 'Count', 'Name']])

# Representative docs for top 3 topics
top_topics = topic_info[topic_info['Topic'] != -1].head(3)['Topic'].tolist()
for tid in top_topics:
    kws = [w for w, _ in topic_model.get_topic(tid)[:5]]
    printmd(f"\n**Topic {tid} keywords:** {', '.join(kws)}")
    reps = topic_model.get_representative_docs(tid)
    for doc in reps[:2]:
        print(f"  · {doc[:200]}")

## 4. Save and publish to SuAVE

In [ ]:
new_file = suaveint.save_csv_file(df, absolutePath, csv_file)

In [ ]:
input_text = widgets.Text(placeholder='Enter survey name...')
output_text = widgets.Text()

def _bind(sender):
    output_text.value = input_text.value

input_text.on_submit(_bind)
printmd("**Enter survey name, press Enter, then run the next cell:**")
display(input_text, output_text)

In [ ]:
survey_name = output_text.value
suaveint.create_survey(survey_url, new_file, survey_name, dzc_file, user, csv_file, view, views)